In [88]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

import torch
from torch import nn

from torch.utils.data import DataLoader

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cuda


In [89]:
DATA_DIR = "../data"
GLOB_DIR = DATA_DIR + '/data_global'
STAT_DIR = DATA_DIR + '/data_stats'

TREE_BAND = ['height', 'agb', 'soil', 'lai', 'gpp', 'npp', 'rh']
AGES = [1, 10, 20, 30, 50, 70, 100, 150, 200, 250, 300, 350, 400, 450, 500]
N_SITE, N_YEAR, N_MONTH, N_FEA, N_OUT = 54152, 40, 12, 136, 7

print('Data dir :', DATA_DIR)
print('Targets  :', TREE_BAND)
print('Seed ages:', AGES)

mask2d = np.load(STAT_DIR + '/mask2d.npy')                # (360, 720), coordinates of land pixels

# Naive evolution of targets with age, based on the mean and slope of each target across age and location.
age_triplet = np.load(STAT_DIR + '/age_triplet.npz')            # (15,), (15,), (15,)
TRI_AGE, TRI_MEAN, TRI_SLOPE = age_triplet['age'], age_triplet['age_mean'], age_triplet['age_slope']

Data dir : ../data
Targets  : ['height', 'agb', 'soil', 'lai', 'gpp', 'npp', 'rh']
Seed ages: [1, 10, 20, 30, 50, 70, 100, 150, 200, 250, 300, 350, 400, 450, 500]


In [90]:
stats = np.load(STAT_DIR + '/data_stats.npz')
x_mean, x_std = stats['x_mean'], stats['x_std'] # stats for normalization of the input variables (features, 136) 
y_mean, y_std = stats['y_mean'], stats['y_std']  # stats for normalization of the output variables (targets, 7)

def normalize_x(x, x_mean=x_mean, x_std=x_std):
    return (x - x_mean) / (x_std + 1e-10)

def inv_x(x, x_mean=x_mean, x_std=x_std):
    return x * (x_std + 1e-10) + x_mean


train_sample = np.load(GLOB_DIR + '/res_train4_test8.npz')
x_train, y_train, x_test, y_test = train_sample['x_train'], train_sample['y_train'], train_sample['x_test'], train_sample['y_test']

y_train = y_train.transpose(1, 0, 2, 3, 4)  # Transpose y_train so that site comes first
y_test = y_test.transpose(1, 0, 2, 3, 4)  # Transpose y_test so that site comes first

x_train, y_train = normalize_x(x_train), normalize_x(y_train, y_mean, y_std)
x_test, y_test = normalize_x(x_test), normalize_x(y_test, y_mean, y_std)


In [91]:
def prep_year(x, y, year, month = 12):
    """
    Prepare the data for a specific year.
    Args:
        x: Input features (N_SITE, N_YEAR, N_MONTH, N_FEA)
        y: Output targets (MODEL_AGE, N_SITE, N_YEAR, N_MONTH, N_OUT)
        year: The year to prepare (0-indexed)
        month: Month to look at (default is 12, comparing december to december)
        norm: Whether to normalize the data (default is True)
    Returns:
        x_year: Input features for the specified year (N_SITE, N_FEA)
        y_year: Output targets for the specified year (MODEL_AGE, N_SITE, N_OUT)
    """

    # FIX: use the annual MEAN of the 12 monthly inputs, not just December's snapshot.
    # The original `x[:, year, month-1, :]` threw away 11 of 12 months of environmental
    # drivers (temperature/precip/radiation/wind/humidity seasonality) -- the tutorial's RF
    # baseline instead averages the whole year (`X[site_idx].mean(axis=2)`) before using it as
    # a feature, since e.g. GPP/NPP/Rh are noted (in the CarbonGlobe paper) to depend heavily on
    # within-year climate variation, not just the December value. `month` is now unused for `x`
    # but kept as an argument since `y` still intentionally uses only December (see below).
    x_year = x[:, year, :, :].mean(axis=1)  # (N_SITE, N_FEA), annual mean over the 12 months

    # y intentionally stays December-only: this matches the tutorial/paper's convention of using
    # December as the "annual" checkpoint for every target, so this is NOT a bug.
    y_year = y[:, :, year, month-1, :]  # (MODEL_AGE, N_SITE, N_OUT)
    return x_year, y_year

input_size = x_train.shape[-1] + 8  # 136 features + 1 for age + previous 7 targets
n_classes = y_train.shape[-1]  # 7 targets

class SimpleNN(nn.Module):
    def __init__(self, input_size, output_size):
        super(SimpleNN, self).__init__()
        self.fc1 = nn.Linear(input_size, 128)
        self.fc2 = nn.Linear(128, 64)
        self.fc3 = nn.Linear(64, output_size)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = self.fc3(x)
        return x

model = SimpleNN(input_size=input_size, output_size=N_OUT).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
loss = nn.MSELoss()


# epochs = 10
# batch_size = 16
# year_to_train = 0  # Change this to the year you want to train on (0-indexed)

# loss_history = []
# val_loss_history = []

# for epoch in range(epochs):
#     model.train()
#     x_year, y_year = prep_year(x_train, y_train, year_to_train)

#     age_feature = torch.full((x_year.shape[0], 1), AGES[year_to_train], dtype=torch.float32)
#     x_year = torch.tensor(x_year, dtype=torch.float32)
#     x_year = torch.cat((x_year, age_feature), dim=1)

#     batch_indices = np.random.choice(x_year.shape[0], batch_size, replace=False)
#     epoch_loss = 0.0
#     for idx in batch_indices:
#         optimizer.zero_grad()
#         output = model(x_year[idx].unsqueeze(0))  # Add batch dimension
#         target = torch.tensor(y_year[:, idx, :], dtype=torch.float32).mean(dim=0)  # Mean across ages for the target
#         loss_value = loss(output, target.unsqueeze(0))  # Add batch dimension to target
#         loss_value.backward()
#         epoch_loss += loss_value.item()
#         optimizer.step()

#     loss_history.append(epoch_loss / batch_size)

#     # Evaluate on validation set
#     model.eval()
#     x_val_year, y_val_year = prep_year(x_test, y_test, year_to_train)
#     age_feature = torch.full((x_val_year.shape[0], 1), AGES[year_to_train], dtype=torch.float32)
#     x_val_year = torch.tensor(x_val_year, dtype=torch.float32)
#     x_val_year = torch.cat((x_val_year, age_feature), dim=1)
#     with torch.no_grad():
#         val_output = model(x_val_year)
#         val_target = torch.tensor(y_val_year, dtype=torch.float32).mean(dim=0)
#         val_loss_value = loss(val_output, val_target.unsqueeze(0))
#         val_loss_history.append(val_loss_value.item())
    
# plt.plot(loss_history, label='Training Loss')
# plt.plot(val_loss_history, label='Validation Loss')
# plt.xlabel('Epoch')
# plt.ylabel('Loss')
# plt.legend()
# plt.show()


In [92]:
def recursive_predict(model, x, y, start_year=0, end_year=39, month=12):
    """
    Recursively predict the targets for each year from start_year to end_year.
    Args:
        model: The trained model
        x: Input features (N_SITE, N_YEAR, N_MONTH, N_FEATS)
        y: Output targets (N_SITE, MODEL_AGE, N_YEAR, N_MONTH, N_OUT)
        start_year: The starting year for prediction (0-indexed)
        end_year: The ending year for prediction (0-indexed)
        month: Month to look at (default is 12, comparing december to december)
    Returns:
        prediction_history: tensor (N_YEARS, AGES, N_SITES, N_OUT), still attached to the
            computation graph so the whole rollout can be backpropagated through.
    """

    prediction_history = []

    y_in = y[:, :, start_year, month-1, :]  # (N_SITE, AGES, N_OUT)
    y_in = np.transpose(y_in, (1, 0, 2))  # (AGES, N_SITE, N_OUT), to match x_year's age-major layout

    y_in = torch.tensor(y_in, dtype=torch.float32, device=device)  # (AGES, N_SITE, N_OUT)
    for year in range(start_year, end_year+1):
        x_year, y_year = prep_year(x, y, year, month)

        x_year = torch.tensor(x_year, dtype=torch.float32, device=device)  # (N_SITE, N_FEATS)
        x_year = x_year.tile(len(AGES), 1, 1)  # (AGES, N_SITES, N_FEATS)

        # FIX: scale age to [0, 1] (age / 500), matching the tutorial baseline (`age / 500.0`).
        # x/y are z-scored (~unit variance); feeding a raw age up to 500 alongside them gives one
        # feature a scale ~2-3 orders of magnitude larger than the rest. That's harmless for the
        # RF baseline (tree splits are scale-invariant) but can distort gradient-based training
        # here, since the optimizer effectively sees a very different implicit step size on this
        # feature vs. the others.
        ages = (torch.tensor(AGES, dtype=torch.float32, device=device) / 500.0).unsqueeze(1).repeat(1, x_year.shape[1]).unsqueeze(2) # (AGES, N_SITES, 1 feature)
        x_year = torch.cat((x_year, ages), dim=2)  # (AGES, N_SITES, N_FEATS + 1); x_year is already a tensor, no need to re-wrap it
        x_year = torch.cat((x_year, y_in), dim=-1)  # (AGES, N_SITES, N_FEATS + N_OUT+1)

        output = model(x_year) # separate predictions for each age

        prediction_history.append(output)  # keep as a tensor so gradients flow through the rollout
        y_in = output  # NOT detached: lets gradients backprop through time across all years
    return torch.stack(prediction_history)  # (N_YEARS, AGES, N_SITES, N_OUT)


class SiteDataset(torch.utils.data.Dataset):
    """Wraps per-site (x, y) arrays for use with a DataLoader."""
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __len__(self):
        return self.x.shape[0]

    def __getitem__(self, idx):
        return self.x[idx], self.y[idx]


epochs = 10
batch_size = 16
month = 12
start_year, end_year = 0, 39

train_dataset = SiteDataset(x_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

# NOTE on training convention: this loop calls the SAME `recursive_predict` for training as for
# testing below -- i.e. training is fully autoregressive (the model is fed its own prediction as
# `v` for all 40 years, gradients flowing back through the whole rollout) rather than the
# tutorial/paper's teacher-forced one-step training (true previous-year `v` given during training,
# autoregressive only at test time). See agent/DATASET_NOTES.md for a longer discussion of this
# difference and a proposed teacher/student setup to bridge it -- this is a deliberate design
# choice worth being aware of, not something fixed here, since both are legitimate strategies.
for i in range(epochs):
    model.train()
    epoch_loss = 0.0
    n_batches = 0

    for x_batch, y_batch in train_loader:
        x_batch = x_batch.numpy()
        y_batch = y_batch.numpy()

        optimizer.zero_grad()
        predictions_tensor = recursive_predict(model, x_batch, y_batch, start_year=start_year, end_year=end_year, month=month)
        # (N_YEARS, AGES, N_SITES, N_OUT)

        # Align the target with the prediction layout: (N_SITE, AGES, N_YEAR, N_OUT) -> (N_YEARS, AGES, N_SITE, N_OUT)
        target_np = y_batch[:, :, start_year:end_year+1, month-1, :]
        target_np = np.transpose(target_np, (2, 1, 0, 3))
        target_tensor = torch.tensor(target_np, dtype=torch.float32, device=device)

        loss_value = loss(predictions_tensor, target_tensor)
        loss_value.backward()
        optimizer.step()

        epoch_loss += loss_value.item()
        n_batches += 1

    print(f"Epoch {i+1}/{epochs}, Loss: {epoch_loss / n_batches:.4f}")

Epoch 1/10, Loss: 0.3515
Epoch 2/10, Loss: 0.1948
Epoch 3/10, Loss: 0.1625
Epoch 4/10, Loss: 0.1467
Epoch 5/10, Loss: 0.1368
Epoch 6/10, Loss: 0.1329
Epoch 7/10, Loss: 0.1259
Epoch 8/10, Loss: 0.1250
Epoch 9/10, Loss: 0.1183
Epoch 10/10, Loss: 0.1134


In [93]:

test_loader = DataLoader(SiteDataset(x_test, y_test), batch_size=batch_size, shuffle=False)

predictions_list = []
targets_list = []

for x_batch, y_batch in test_loader:
    model.eval()
    x_batch = x_batch.numpy()
    y_batch = y_batch.numpy()

    predictions_tensor = recursive_predict(model, x_batch, y_batch, start_year=start_year, end_year=end_year, month=month)
    # (N_YEARS, AGES, N_SITES, N_OUT)

    # Align the target with the prediction layout: (N_SITE, AGES, N_YEAR, N_OUT) -> (N_YEARS, AGES, N_SITE, N_OUT)
    target_np = y_batch[:, :, start_year:end_year+1, month-1, :]
    target_np = np.transpose(target_np, (2, 1, 0, 3))
    target_tensor = torch.tensor(target_np, dtype=torch.float32, device=device)

    predictions_list.append(predictions_tensor.cpu().detach().numpy())
    targets_list.append(target_tensor.cpu().numpy())

In [94]:
def calc_eval_metrics(prediction_history, targets):
    """
    Calculate evaluation metrics (RMSE, MAE, R2) for each feature prediction.
    Args:
        prediction_history: tensor (N_YEARS, AGES, N_SITES, N_OUT)
        targets: tensor (N_YEARS, AGES, N_SITES, N_OUT)
    Returns:
        rmse: total rmse
        mae: total mae
        r2: total r2
        delta: RMSE of the error in year-over-year change (predicted change vs. true change)
        ce: rmse of the final step
    """

    prediction_history = prediction_history.detach().cpu().numpy()
    targets = targets.detach().cpu().numpy()

    rmse = np.sqrt(np.mean((prediction_history - targets) ** 2, axis=(0, 1, 2)))  # RMSE for each output feature
    mae = np.mean(np.abs(prediction_history - targets), axis=(0, 1, 2))
    
    # R2 score
    ss_res = np.sum((targets - prediction_history) ** 2, axis=(0, 1, 2))
    ss_tot = np.sum((targets - np.mean(targets, axis=(0, 1, 2), keepdims=True)) ** 2, axis=(0, 1, 2))
    r2 = 1 - (ss_res / ss_tot)

    # FIX: delta error must compare the PREDICTED year-over-year change against the TRUE
    # year-over-year change, not just measure the predictions' own volatility in isolation.
    # The previous version (`prediction_history[1:] - prediction_history[:-1]`, no `targets`
    # involved) never touched the ground truth, so it wasn't measuring error at all -- a model
    # that swings wildly or barely moves, right or wrong, would just report its own volatility.
    # This matches the tutorial/paper's E_delta = RMSE((Y_t - Y_t-1) - (Yhat_t - Yhat_t-1)).
    dt = targets[1:] - targets[:-1]
    dp = prediction_history[1:] - prediction_history[:-1]
    delta = np.sqrt(np.mean((dp - dt) ** 2, axis=(0, 1, 2)))

    # RMSE of the final step
    ce = np.sqrt(np.mean((prediction_history[-1] - targets[-1]) ** 2, axis=(0, 1)))

    return rmse, mae, r2, delta, ce


# convert predictions and targets back out of normalized space for evaluation

predictions_vals = inv_x(np.concatenate(predictions_list, axis=2), y_mean, y_std)
targets_vals = inv_x(np.concatenate(targets_list, axis=2), y_mean, y_std)

metrics = np.array(calc_eval_metrics(torch.tensor(predictions_vals), torch.tensor(targets_vals)))

labels = ['height', 'agb', 'soil', 'lai', 'gpp', 'npp', 'rh']

for i, label in enumerate(labels):
    print(f"{label}: RMSE={metrics[0][i]:.4f}, MAE={metrics[1][i]:.4f}, R2={metrics[2][i]:.4f}, Delta={metrics[3][i]:.4f}, CE={metrics[4][i]:.4f}")

height: RMSE=4.4185, MAE=3.2240, R2=0.8199, Delta=0.8558, CE=4.3881
agb: RMSE=1.8668, MAE=1.0840, R2=0.8763, Delta=0.4582, CE=2.0061
soil: RMSE=2.5137, MAE=1.7200, R2=0.8350, Delta=0.4009, CE=2.5390
lai: RMSE=0.7062, MAE=0.4405, R2=0.8741, Delta=0.4359, CE=0.7190
gpp: RMSE=0.4264, MAE=0.2320, R2=0.8974, Delta=0.3388, CE=0.4087
npp: RMSE=0.2101, MAE=0.1120, R2=0.8942, Delta=0.1696, CE=0.1981
rh: RMSE=0.2123, MAE=0.1272, R2=0.8830, Delta=0.1913, CE=0.2167
